# Cable-Driven Finger

This notebook simulates a finger that is closed by two cables and opened by return springs, and checks the result against a hand calculation.

The finger has three hinge joints, each with a return spring with preload. Two cables close it, each pulled by its own linear actuator:

- the **knuckle cable** runs over a sheave on the first joint and ends on the first link: it bends the first joint only;
- the **curl cable** runs over a pulley at every joint to the fingertip. On the first joint it passes a small idler, so it bends the two outer joints long before it moves the first one.

With two tensions the finger takes poses that one cable cannot give: a bent first joint with straight outer joints, a hook (outer joints curled on a straight first joint), and anything between.

You will:

1. Compute the joint angles and the travel of both actuators for a few pairs of cable tensions by hand (closed form).
2. Build the finger with two massless routed tendons and step it with `SolverVBD` on the CPU.
3. Compare the simulated angles and travels with the closed form.
4. See what changes for a rod cable (a short section; the rod step itself needs CUDA and is switched off).

The finger, its springs and its actuators come from the helper module `newton/examples/cable/cable_hand.py`, which also backs the example `python -m newton.examples cable_driven_finger`. The page [Cable-Driven Mechanisms](../concepts/cable_driven_mechanisms.rst) explains the mechanism, the two cable models and their settings.

## Setup and Imports

In [ ]:
# Colab setup: install Newton before importing it.
import importlib.util
import subprocess
import sys

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "newton[notebook]"])

In [ ]:
import numpy as np
import warp as wp

import newton
from newton.examples.cable import cable_hand

wp.config.log_level = wp.LOG_WARNING

## The Hand Calculation

`default_hand_spec()` returns every number of the design: link lengths, sheave radii, springs, cables and actuators. We use the index finger, which has two cables.

Without friction a cable carries its actuator tension along its whole length. Cable $c$ acts on joint $j$ with the moment arm $r_{cj}$, the radius of its pulley on that joint (zero where the cable does not pass the joint). The torque balance of joint $j$, with spring preload $\tau_{0,j}$ and spring stiffness $k_j$, gives

$$q_j(T) = \mathrm{clip}\left(\frac{\sum_c r_{cj}\,T_c - \tau_{0,j}}{k_j},\ 0,\ q_{\max,j}\right).$$

The cell below prints the moment arms. The knuckle cable has an arm on the first joint only. The curl cable has a small arm on the first joint (the idler), so the curl cable also adds a little torque there: this cross-term is why the two cables are not fully independent.

In [ ]:
spec = cable_hand.default_hand_spec()
digit = spec.digits["index"]
cables = digit.cable_names  # ["curl", "knuckle"]
arms = cable_hand.moment_arm_matrix(digit)  # [m], one row per cable, one column per joint

print("cable    arm on joint 1, 2, 3 [mm]")
for name, row in zip(cables, arms, strict=True):
    print(f"{name:8s} " + "  ".join(f"{1e3 * r:5.1f}" for r in row))

print("\njoint  spring preload [mN*m]  stiffness [mN*m/rad]  upper stop [deg]")
for j, joint in enumerate(digit.joints):
    print(
        f"{j + 1:5d}  {1e3 * joint.spring_preload:21.1f}  {1e3 * joint.spring_stiffness:20.1f}"
        f"  {np.degrees(joint.upper):15.0f}"
    )

`frictionless_row` evaluates the closed form at one pair of tensions. Besides the joint angles it returns the travel of both actuators: the cable taken up by the pulleys, $\sum_j r_{cj} q_j$, plus the stretch of the cable, of its anchors and of the hinges it passes. Travel is measured from the state with both cables at the rest tension (2 N). Because the curl cable passes the first joint on its idler, its actuator also travels when only the knuckle cable bends that joint.

`two_cable_poses` names a few useful pairs of tensions. We take three: the first joint bent with straight outer joints, a hook, and both combined.

In [ ]:
poses = cable_hand.two_cable_poses(spec, "index")
labels = ["knuckle_only", "hook", "pinch_ready"]
tensions = [poses[label] for label in labels]  # each a dict: {"curl": [N], "knuckle": [N]}
reference = [cable_hand.frictionless_row(spec, "index", tension) for tension in tensions]

print("pose          curl / knuckle [N]   joint angles [deg]     travel curl / knuckle [mm]")
for label, row in zip(labels, reference, strict=True):
    angles = "  ".join(f"{np.degrees(q):6.2f}" for q in row["q"])
    travel = " / ".join(f"{1e3 * v:6.3f}" for v in row["travel"])
    print(f"{label:13s} {row['T'][0]:5.1f} / {row['T'][1]:5.1f}        {angles}   {travel}")

## Build the Finger with Massless Tendons

`build_tendon_finger_cables_model` creates a fixed base, three links on hinge joints, one actuator carriage per cable on a prismatic joint, and one routed tendon per cable from its carriage over the guides and pulleys to its anchor. A tendon has no mass and no collision shapes.

Three details matter for `SolverVBD`:

- **Springs.** The stiffness of a return spring is a joint drive whose target is the lower joint limit. The preload cannot be part of that drive, because drive targets are clamped to the joint limits. It is a constant joint force, written once into `Control.joint_f` by `create_spring_preload`.
- **Gravity.** The model is built without gravity, and `create_body_weight` adds the weights as body forces in every substep. The solver created by `make_tendon_solver` takes external forces inside its solve, which converges on these light links.
- **Actuators.** `create_actuator_governor` moves the target of every carriage drive in every substep so that the drive force stays between the rest tension and the force limit and the target moves no faster than the speed limit. The command is one cable tension per actuator, in the order of `governor.actuators`.

The flexion axis is the world z axis, so gravity does not load the joints and the closed form above applies.

In [ ]:
gravity = (0.0, 0.0, -cable_hand.GRAVITY)

model, skeleton, tendons = cable_hand.build_tendon_finger_cables_model(
    spec, "index", gravity=(0.0, 0.0, 0.0), device="cpu"
)
solver = cable_hand.make_tendon_solver(model)

state_0, state_1 = model.state(), model.state()
control = model.control()
cable_hand.create_spring_preload(model, skeleton).apply(control)
weight = cable_hand.create_body_weight(model, gravity)
governor = cable_hand.create_actuator_governor(model, spec, skeleton)

links = [tendon.link_count for tendon in tendons]
print(f"bodies: {model.body_count}, joints: {model.joint_count}, tendon links per cable: {links}")

## Step Through Three Poses

One substep clears the body forces, adds the weights, updates the command and the governor and steps the solver. The tendon settings are tuned for 20 substeps per 60 Hz frame.

The commands move from pose to pose smoothly (`create_staircase_command`), not in steps. After each pose we read the joint angles and the travel of both carriages with `newton.eval_ik`, and the tension at both ends of each tendon. We also keep the lowest tension seen on the way: no cable should come close to slack.

In [ ]:
fps = cable_hand.TENDON_FPS
substeps = cable_hand.TENDON_SUBSTEPS
dt = 1.0 / (fps * substeps)

# The governor moves the reference of the knuckle actuator at 28 mm/s at most and that of the curl actuator
# at 24 mm/s (the curl cable is ramped more slowly so that the knuckle cable stays taut).
rates = np.array([cable_hand.actuator_reference_rate(spec, "index", name) for name in cables])
print("reference rate [mm/s]:", ", ".join(f"{name} {1e3 * rate:.0f}" for name, rate in zip(cables, rates, strict=True)))

# The tension commands are not stepped. A staircase command blends them from pose to pose over 0.6 s, the
# way the keyframe scripts of the hand do; a stepped knuckle command would let the curl cable dip toward
# slack. One period is the time for the largest travel step, half the blend, and a hold of 0.3 s.
rest = spec.cable.tension_rest
levels = [[rest] * len(cables)] + [row["T"] for row in reference]  # start at rest, then the three poses
travels = np.array([[0.0] * len(cables)] + [row["travel"] for row in reference])
largest = float((np.abs(np.diff(travels, axis=0)) / rates).max())
hold = 0.3
frames = int(np.ceil((largest + 0.5 * cable_hand.STAIRCASE_BLEND + 0.15 + hold) * fps))  # frames per pose
staircase = cable_hand.create_staircase_command(
    spec, levels=levels, hold=hold, transition=frames / fps - hold, device="cpu"
)
print(f"command blend {staircase.blend:.1f} s, one pose every {frames / fps:.2f} s")


def simulate_frame():
    """Advance the tendon finger by one frame."""
    global state_0, state_1
    for _ in range(substeps):
        state_0.clear_forces()
        weight.apply(state_0)
        staircase.step(governor.command, dt)
        governor.step(state_0, control, dt)
        solver.step(state_0, state_1, control, None, dt)
        state_0, state_1 = state_1, state_0


def read_pose(model, state, skeleton):
    """Joint angles [rad] and the travel of every carriage [m] of a state."""
    joint_q, joint_qd = wp.clone(model.joint_q), wp.clone(model.joint_qd)
    newton.eval_ik(model, state, joint_q, joint_qd)
    joint_q = joint_q.numpy()
    return joint_q[list(skeleton.q_flexion)], joint_q[list(skeleton.q_sliders)]


for _ in range(frames):  # the first level: both cables at the rest tension
    simulate_frame()
results = []
lowest = np.full(len(cables), np.inf)
for label in labels:
    for _ in range(frames):
        simulate_frame()
        tension = [cable_hand.tendon_span_tensions(solver, tendon) for tendon in tendons]
        lowest = np.minimum(lowest, [float(t.min()) for t in tension])
    q, travel = read_pose(model, state_0, skeleton)
    ends = [cable_hand.tendon_end_tensions(solver, tendon) for tendon in tendons]
    results.append((q, travel))
    measured = ", ".join(f"{name} {a:5.2f} / {b:5.2f} N" for name, (a, b) in zip(cables, ends, strict=True))
    print(f"{label:13s} tension at the actuator / at the far end: {measured}")
print("lowest tension on the way:", ", ".join(f"{name} {t:.2f} N" for name, t in zip(cables, lowest, strict=True)))
assert lowest.min() > 0.5

## Compare with the Closed Form

The tendon finger follows the hand calculation closely: a few hundredths of a degree in the joint angles and a few micrometers in the travels.

In [ ]:
print("pose          simulated angles [deg]  closed form [deg]       travel sim / closed form [mm] (curl, knuckle)")
for label, row, (q, travel) in zip(labels, reference, results, strict=True):
    simulated = "  ".join(f"{np.degrees(v):6.2f}" for v in q)
    expected = "  ".join(f"{np.degrees(v):6.2f}" for v in row["q"])
    travels = ", ".join(f"{1e3 * a:6.3f} / {1e3 * b:6.3f}" for a, b in zip(travel, row["travel"], strict=True))
    print(f"{label:13s} {simulated}  {expected}  {travels}")

angle_error = max(
    float(np.abs(np.degrees(q - np.array(row["q"]))).max()) for row, (q, _) in zip(reference, results, strict=True)
)
travel_error = max(
    float(np.abs(travel - np.array(row["travel"])).max()) for row, (_, travel) in zip(reference, results, strict=True)
)
print(f"\nlargest angle error: {angle_error:.3f} deg, largest travel error: {1e3 * travel_error:.3f} mm")
assert angle_error < 0.3 and travel_error < 1.0e-4

The closed form over both tensions shows what each cable does. Along a row the curl cable closes the two outer joints; down a column the knuckle cable closes the first joint. The first joint also moves a little along a row: that is the curl cable's small arm on the idler.

To hold a first-joint angle while the curl tension changes, lower the knuckle tension by the curl cable's share: `cable_tensions_for_knuckle` does that.

In [ ]:
curl_levels = [2.0, 6.0, 10.0, 14.0, 18.0]  # [N]
print("joint angles [deg] (1, 2, 3)")
print("knuckle [N] \\ curl [N]  " + "".join(f"{t:^22.0f}" for t in curl_levels))
for knuckle in (2.0, 12.0, 20.0, 28.0):
    cells = []
    for curl in curl_levels:
        q = cable_hand.frictionless_angles(digit, {"curl": curl, "knuckle": knuckle})
        cells.append(" ".join(f"{np.degrees(v):6.1f}" for v in q))
    print(f"{knuckle:21.0f}  " + "  ".join(cells))

# the same first-joint angle of 45 deg at two curl tensions
for curl in (2.0, 14.0):
    pair = cable_hand.cable_tensions_for_knuckle(spec, "index", curl, np.radians(45.0))
    q = cable_hand.frictionless_angles(digit, pair)
    print(f"curl {curl:4.1f} N: knuckle {pair['knuckle']:5.2f} N holds the first joint at {np.degrees(q[0]):.1f} deg")

## Now with Rod Cables

The same finger can carry rod cables: every cable a chain of 1 mm capsule bodies that is kept on flanged pulleys by contact only. The skeleton, the springs, the weights and the governor are the same. What changes:

- `build_rod_finger_cables_model` adds the rod segments of both cables and the pulley collision shapes (a few hundred bodies instead of six).
- A collision pipeline finds the contacts between cables and pulleys in every substep. Create it before the solver.
- `create_rod_solver` uses the block-sparse articulation solve, 30 substeps per frame and more iterations. It takes the reference inertia of the rod segments from `rod_inertia_reference` and a contact buffer sized by `rod_contact_buffer`.
- Before anything is compared, both cables are run once to 16 N and back (a seating stroke), so that every rod lies on its pulleys.

The rod cables are meant for CUDA, where one frame can be captured as a graph. The cell below builds the rod finger and holds the rest tension for half a second. It is switched off by default: set `RUN_ROD_CABLE = True` to run it on a machine with a CUDA device.

In [ ]:
RUN_ROD_CABLE = False  # set to True to build and hold the rod finger (needs a CUDA device)

if RUN_ROD_CABLE and wp.is_cuda_available():
    device = wp.get_cuda_device()
    rod_model, rod_skeleton, rods = cable_hand.build_rod_finger_cables_model(
        spec, "index", gravity=(0.0, 0.0, 0.0), device=device
    )
    rod_dt = 1.0 / (fps * cable_hand.ROD_SUBSTEPS)
    pipeline = cable_hand.create_rod_collision_pipeline(rod_model)
    contacts = pipeline.contacts()
    rod_solver = cable_hand.create_rod_solver(
        rod_model,
        rod_dt,
        inertia_reference=cable_hand.rod_inertia_reference(spec),
        rigid_body_contact_buffer_size=cable_hand.rod_contact_buffer(spec, "index"),
    )

    rod_states = [rod_model.state(), rod_model.state()]
    rod_control = rod_model.control()
    cable_hand.create_spring_preload(rod_model, rod_skeleton).apply(rod_control)
    rod_weight = cable_hand.create_body_weight(rod_model, gravity)
    rod_governor = cable_hand.create_actuator_governor(rod_model, spec, rod_skeleton)

    # One frame as a CUDA graph. The even substep count leaves the two states in their starting roles.
    with wp.ScopedCapture(device=device) as capture:
        for _ in range(cable_hand.ROD_SUBSTEPS):
            rod_states[0].clear_forces()
            rod_weight.apply(rod_states[0])
            rod_governor.step(rod_states[0], rod_control, rod_dt)
            pipeline.collide(rod_states[0], contacts)
            rod_solver.step(rod_states[0], rod_states[1], rod_control, contacts, rod_dt)
            rod_states.reverse()

    for _ in range(round(0.5 * fps)):  # rest tension: the cables take up their slack
        wp.capture_launch(capture.graph)
    q, _ = read_pose(rod_model, rod_states[0], rod_skeleton)

    print(f"rod finger: {rod_model.body_count} bodies, {[len(rod.bodies) for rod in rods]} rod bodies per cable")
    print("  joint angles at the rest tension [deg]:", "  ".join(f"{np.degrees(v):6.2f}" for v in q))
elif RUN_ROD_CABLE:
    print("No CUDA device: the rod cable cell is skipped.")
else:
    print("The rod cable cell is switched off; set RUN_ROD_CABLE = True to run it.")

A rod finger follows the same closed form less closely than the tendon finger. In return it resolves the contact between cable and pulleys.

## Next Steps

- Run `python -m newton.examples cable_driven_finger` to watch the finger go through its poses. The two cables are drawn in different colors (curl cable cyan-blue, knuckle cable magenta-pink) and brighten with their tension. Add `--xray` to see the cable paths, `--tension-pairs` for your own sequence of tensions, `--digit ring` for a finger with one cable, or `--cable-model rod` for the rod cables.
- Run `python -m newton.examples cable_driven_hand` for a whole hand with seven cables.
- Read [Cable-Driven Mechanisms](../concepts/cable_driven_mechanisms.rst) for the routing rules, the closed form with friction, and the solver settings of both cable models.